## NER & SA

In [1]:
import pandas as pd
import spacy
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [2]:
content = "Trinamool Congress leader Mahua Moitra has moved the Supreme Court against her expulsion from the Lok Sabha over the cash-for-query allegations against her. Moitra was ousted from the Parliament last week after the Ethics Committee of the Lok Sabha found her guilty of jeopardising national security by sharing her parliamentary portal's login credentials with businessman Darshan Hiranandani."

In [3]:
nlp = spacy.load("en_core_web_sm")

In [4]:
doc = nlp(content)

for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

Trinamool Congress 0 18 ORG
Mahua Moitra 26 38 PERSON
the Supreme Court 49 66 ORG
Moitra 157 163 NORP
Parliament 184 194 ORG
last week 195 204 DATE
the Ethics Committee 211 231 ORG
Darshan Hiranandani 373 392 PERSON


In [5]:
text = "TextBlob is very amazing and simple to use. What a great tool!"

In [6]:
blob = TextBlob(text)
blob.sentiment

Sentiment(polarity=0.5933333333333334, subjectivity=0.7023809523809524)

In [7]:
sid_obj = SentimentIntensityAnalyzer()
sentiment_dict = sid_obj.polarity_scores(text)

## BERT & GPT

In [8]:
df = pd.read_csv("../../data/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [10]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification


In [11]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
review = df['review'].iloc[0]

inputs = tokenizer(review, return_tensors="pt", truncation=True, padding=True)

outputs = model(**inputs)
logits = outputs.logits
pred = torch.argmax(logits, dim=1)

print(review[:120])
print(pred.item())

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this
0


In [18]:
from datasets import Dataset
from transformers import Trainer, TrainingArguments

In [19]:
df = df.sample(500)
df['label'] = df['sentiment'].map({"postive":1, "negative":0})
dataset = Dataset.from_pandas(df[['review', 'label']])


In [20]:
def tokenize(batch):
    return tokenizer(batch['review'], truncation=True, padding='max_length', max_length=256)

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=8,
)

trainer = Trainer(
    model=model,
    args = training_args,
    train_dataset=dataset
)

trainer.train()

/home/asish-jose/AI-ML/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [4]:
import torch
print(torch.cuda.is_available())

device = torch.device("cpu")
model.to(device)

False


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name='gpt2'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
prompt = ""

inputs = tokenizer(prompt, return_tensors="pt").to(device)

ouput = model.generate(
    **inputs,
    max_length=50,
    do_sample=True,
    temperature=1.2,
    top_k=50,
    top_p=0.9,
    num_return_sequences=3
)

output = tokenizer.decoder(output[0], skip_special_tokens=True)

## lstm

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

with open(".......") as f:
    text = f.read().lower()[:20000]

In [ ]:
words = text.lower().split()

vocab = sorted(set(words))
stoi = {w:i for i,w in enumerate(vocab)}
vocab_size = len(vocab)

data = torch.tensor([stoi[w] for w in words])

seq_len = 10
X, Y = [], []
for i in range(len(data)-seq_len):
    X.append(data[i:i+seq_len])
    Y.append(data[i+1:i+seq_len+1])

X = torch.stack(X)
Y = torch.stack(Y)

loader = DataLoader(TensorDataset(X, Y), batch_size=32)

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__():
        self.embed = nn.Embedding(vocab_size, 64)
        self.lstm = nn.LSTM(64, 128, batch_first=True)
        self.fc = nn.Linear(128, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        out,_ = self.lstm(x)
        return self.fc(out)
    
model = LSTMModel()
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.001)

for X,Y in loader:
    opt.zero_grad()
    out = model(X)
    loss = loss_fn(out, Y)
    loss.backward()
    opt.step()

## Translation Model

In [ ]:
class TranslationModel(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.src_emb = nn.Embedding(len(src_vocab), 32)
        self.tgt_emb = nn.Embedding(len(tgt_vocab), 32)

        self.transformer = nn.Transformer(
            d_model = 32,
            nhead=2,
            num_encoder_layers=1,
            num_decoder_layers=1
        )

        self.fc = nn.Linear(32, len(tgt_vocab))

    def forward(self, src, tgt):
        src = self.src_emb(src).permute()
        tgt = self.tgt_emb(tgt).permute()

        out = self.transformer(src, tgt)
        return self.fc(out)

model = TranslationModel()
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters())